In [ ]:
import pandas as pd
import re
import torch

#libraries for data explore
import seaborn as sns
import matplotlib.pyplot as plt

#libraries for Model training and evaluation
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
CSV_PATH = "/content/drive/MyDrive/phishing_project/datasets/Phishing_Email.csv"

In [ ]:
import pandas as pd
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
df.shape

In [ ]:
# Randomly sample 5,000 rows from the full dataset
df = df.sample(n=5000, random_state=42).reset_index(drop=True)

print(df.head())

In [ ]:
# remove extra unnamed column if it exists
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

# rename text column
df = df.rename(columns={'Email Text': 'text'})

# create label column
df['label'] = df['Email Type'].map({
    'Safe Email': 0,
    'Phishing Email': 1
})

# optional: drop old column
df = df.drop(columns=['Email Type'])

print(df.head())

In [ ]:
# Remove nulls
df.dropna(inplace=True)

# Ensure label is integer
df['label'] = df['label'].astype(int)


In [ ]:
print(df.info())

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

Explore the dataset
Plot Class Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='label') # counts the number of occurrences of each unique value in a column
plt.title('Phishing vs Legitimate Emails')
plt.xticks([0, 1], ['Legitimate', 'Phishing'])
plt.ylabel('Count')
plt.show()

Email/Text special character Length Distribution by Class

This graph is a boxplot comparing the number of special characters used in legitimate vs phishing emails.

Special characters: ! @ # $ % ^ & * ( ) _ + = { } [ ] : ; " ' < > / ? , . ~ etc.:

In [ ]:
df['special_chars'] = df['text'].apply(lambda x: sum(not c.isalnum() and not c.isspace() for c in x))

plt.figure(figsize=(8,5))
sns.boxplot(x='label', y='special_chars', data=df, hue='label', palette='coolwarm', dodge=False)

plt.xticks([0, 1], ['Legitimate', 'Phishing'])
plt.title('Special Characters Count by Class')
plt.xlabel('Label')
plt.ylabel('Special Characters Count')
plt.grid(True)
plt.show()

Most Common Words


In [ ]:
from collections import Counter
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')


stop_words = set(stopwords.words('english'))  #e.g., "the", "is", "and"

# Combine all phishing emails: Joins all texts into one string → converts to lowercase → splits into words
phishing_words = ' '.join(df[df['label']==1]['text']).lower().split() #Selects only phishing emails (where label == 1)

filtered_words = [
    word for word in phishing_words
    if word.isalpha() and word not in stop_words and len(word) > 2
]

word_freq = Counter(filtered_words).most_common(20)

# Barplot of top words
words, counts = zip(*word_freq)  #Unpacks the 20 most common word–count pairs into two separate lists: words and counts
plt.figure(figsize=(10,6))
sns.barplot(x=list(counts), y=list(words),hue =list(counts), palette='magma',legend=False)
plt.title('Top 20 Words in Phishing Emails')
plt.xlabel('Frequency')
plt.show()

Purpose of Word Cloud

 It is used to visually show the most frequent words in phishing emails

In [ ]:
from wordcloud import WordCloud

# Join all filtered phishing words into a single string
wordcloud_text = ' '.join(filtered_words)

# Generate word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='magma').generate(wordcloud_text)

# Display the word cloud
plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Phishing Email Words')
plt.show()

 Split the Dataset

This step splits the dataset into training (70%), validation (15%), and testing (15%) sets using stratified sampling.



In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Tokenization using BERT Tokenizer

These encodings are the tokenized versions of your input texts, structured in a way that BERT can understand

Create train & validation split

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

Convert to list

In [ ]:
train_texts = train_texts.tolist()
val_texts = val_texts.tolist()

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

In [ ]:
print(train_encodings['input_ids'][0])          # View input IDs of the first text
print(train_encodings['attention_mask'][0])

Data input format for bert
input_ids: Token IDs representing the input text.

attention_mask: Binary mask to distinguish real tokens (1) from padding (0).

labels: True class values used for supervised learning.

[CLS]: Special token added at the start of input for classification tasks.

[SEP]: Separator token marking the end of a sentence or segment


PyTorch: Python library used for machine learning and deep learning.

In [ ]:
# Create a custom dataset class for PyTorch
class EmailDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        # Store tokenized inputs (input_ids, attention_mask)
        self.encodings = encodings

        # Store corresponding labels (0 = safe, 1 = phishing)
        self.labels = labels

    def __getitem__(self, idx):
        # Get one sample (input_ids, attention_mask) at index idx
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

        # Add label for that sample
        item['labels'] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        # Return total number of samples in dataset
        return len(self.labels)


# Create training dataset (used to train the model)
train_dataset = EmailDataset(train_encodings, train_labels)

# Create validation dataset (used to evaluate model performance)
val_dataset = EmailDataset(val_encodings, val_labels)

Load Pretrained BERT for Classification

In [ ]:
# Load pre-trained BERT model for text classification
# "bert-base-uncased" is used as the base model
# num_labels=2 indicates binary classification (0 = Safe Email, 1 = Phishing Email)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Define evaluation metrics function


In [ ]:
# Define evaluation metrics function
# This function calculates model performance using accuracy, precision, recall, and F1-score
def compute_metrics(pred):

    # Get true labels from predictions
    labels = pred.label_ids

    # Get predicted labels (choose class with highest probability)
    preds = pred.predictions.argmax(-1)

    # Calculate precision, recall, and F1-score for binary classification
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

    # Calculate accuracy
    acc = accuracy_score(labels, preds)

    # Return all evaluation metrics as a dictionary
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

Training Arguments

TrainingArguments is a configuration class that defines how your model should be trained. You pass it key training parameters such as:

output_dir: Where to save model checkpoints.
learning_rate: Controls how fast the model learns.

num_train_epochs: Number of complete passes through the training data.

evaluation_strategy: When to run validation (e.g., "epoch" means after every epoch).

per_device_train_batch_size: Batch size per GPU/CPU device during training.

logging_dir: Directory to store training logs.

It does not train the model — it only stores training settings that the Trainer will use.

In [ ]:
# Define training settings for the BERT model
training_args = TrainingArguments(
    output_dir='./results',            # Folder to save model checkpoints
    eval_strategy="epoch",             # Evaluate model after each epoch
    save_strategy="epoch",             # Save checkpoint after each epoch
    save_total_limit=2,                # Keep only the latest 2 checkpoints
    learning_rate=2e-5,                # Learning rate for fine-tuning BERT
    per_device_train_batch_size=8,    # Training batch size
    per_device_eval_batch_size=8,     # Evaluation batch size
    num_train_epochs=5,                # Number of full training cycles
    weight_decay=0.01,                 # Regularization to reduce overfitting
    optim="adamw_torch",               # Optimizer used for training
    logging_dir='./logs',              # Folder to save training logs
    logging_strategy="epoch",          # Log results after each epoch
    load_best_model_at_end=True,       # Load best model after training ends
    metric_for_best_model="f1",        # Select best model based on F1-score
    greater_is_better=True             # Higher F1-score means better model
)

Convert Labels to List

In [ ]:
# Convert labels from pandas Series to Python list (fix indexing issue)
train_labels = train_labels.tolist()
val_labels = val_labels.tolist()

Create Dataset Objects

In [ ]:
# Create PyTorch datasets for training and validation
train_dataset = EmailDataset(train_encodings, train_labels)
val_dataset = EmailDataset(val_encodings, val_labels)

Trainer

Trainer is the training engine provided by Hugging Face. It handles the full model training lifecycle. Specifically, it:

Loads and trains your model.

Handles batching, shuffling, and tokenization.

Trains the model using PyTorch under the hood.

Evaluates the model on your eval_dataset.

Uses the settings defined in TrainingArguments.

Automatically supports training on GPU or TPU if available.

Trainer simplifies the process so you don’t need to manually write training loops, backpropagation, or optimizer logic.

In [ ]:
# Create Trainer object to handle training and evaluation of the model
trainer = Trainer(
    model=model,                          # BERT model for classification
    args=training_args,                   # Training configuration (epochs, batch size, etc.)
    train_dataset=train_dataset,          # Dataset used for training the model
    eval_dataset=val_dataset,             # Dataset used for validation during training
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),  # Handles dynamic padding of input data
    compute_metrics=compute_metrics       # Function to calculate accuracy, precision, recall, and F1-score
)

In [ ]:
# Start training the model
trainer.train()

In [ ]:
model.save_pretrained("phishing_model")
tokenizer.save_pretrained("phishing_model")

In [ ]:
import torch
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def predict(email):
    inputs = tokenizer(email, return_tensors="pt", truncation=True, padding=True).to(device)
    outputs = model(**inputs)
    pred = outputs.logits.argmax().item()
    return "🚨 Phishing Email" if pred == 1 else "✅ Safe Email"

iface = gr.Interface(fn=predict, inputs="text", outputs="text")
iface.launch()

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
!pip install nbconvert
!jupyter nbconvert --ClearOutputPreprocessor.enabled=True --inplace phishingdetectorBert.ipynb

In [ ]:
!ls
!pwd
!find / -maxdepth 3 -name "*.ipynb" 2>/dev/null

In [ ]:
notebook_path = "phisingdetectorBert.ipynb"

In [ ]:
import json

# Save current notebook first
from google.colab import _message
nb = _message.blocking_request('get_ipynb')['ipynb']

# Remove widget metadata
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

# Save clean file
with open("clean_notebook.ipynb", "w") as f:
    json.dump(nb, f)

print("Clean notebook saved")